# 기대 수명 예측하기 ② 모델 학습과 성능 개선 (교과서 197~201쪽)

교과서 소스 코드 노트북(191~200쪽)의 뒷부분을 수업용으로 옮긴 사본이에요.

- 위에서부터 셀을 하나씩 실행해요. (셀 왼쪽 ▶ 버튼 또는 **Shift + Enter**)
- 데이터 파일을 인터넷 주소에서 바로 읽으므로 **파일을 올리지 않아도** 돼요.
- 처음 실행할 때 「경고: 이 노트북은 Google에서 작성하지 않았습니다」 창이 뜨면 **「무시하고 계속하기」**를 눌러요.
- 교과서 코드에 더한 줄·셀에는 `(사본에 더함)`이라고 적어 두었어요.

## 준비 · 데이터 수집과 전처리 (190~196쪽에서 한 것을 한 셀로)
데이터를 불러오고, 결측치를 지우고, `Status`를 0과 1로 바꿔요.

> 분홍색 상자에 `FutureWarning`이 뜰 수 있어요. **오류가 아니에요.** 마지막 줄에 `(1649, 22)`가 나오면 성공이에요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u4-structured-1/assets/life-expectancy-data.csv')
# 교과서: df = pd.read_csv('Life Expectancy Data.csv')  ← 파일을 올렸을 때
df = df.dropna()    # 결측치 데이터 삭제
df['Status'] = df['Status'].astype(object)
df = df.replace({'Developing': 0, 'Developed': 1})    # 개발도상국은 0, 선진국은 1
df['Status'] = df['Status'].astype(int)    # (사본에 더함) 판다스 버전과 상관없이 0과 1을 숫자로
df.shape

## [5] 기계학습 유형과 알고리즘 선정
- ① 기계학습 유형: 기대 수명이 몇 세인지를 출력하는 인공지능이므로 **지도학습의 예측**에 해당한다.
- ② 알고리즘: 지도학습의 예측 알고리즘으로는 **선형 회귀**가 대표적이다.

### ③ 훈련 데이터와 테스트 데이터 분리하기
'Life expectancy'와 'Country'를 제외한 속성은 X, 'Life expectancy'는 y에 저장해요.

In [ ]:
# 훈련 데이터와 테스트 데이터로 분리
X = df.drop(['Life expectancy', 'Country'], axis=1)
y = df.loc[:, 'Life expectancy']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,
                                                    random_state = 96)

### ④ 훈련 데이터와 테스트 데이터가 잘 나눠졌는지 확인하기

In [ ]:
X_train.head()

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

### 해 보기 · 비율을 다르게 하여 나누기 (사본에 더한 셀)
`test_size`를 0.1, 0.3, 0.5 등으로 바꿔 실행해 보세요. 아래 단계가 교과서와 같은 결과가 나오도록 **이름을 다르게**(X_tr2 …) 저장해요.

In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size = 0.3, random_state = 96)
print(X_tr2.shape, X_te2.shape)
# (선택) [6]·[7] 셀을 한 번 실행한 뒤, 아래 두 줄 앞의 # 을 지우고 다시 실행하면 이 비율의 결정 계수도 볼 수 있어요.
# m2 = clone(linear_reg).fit(X_tr2, y_tr2)    # clone: [6]에서 만든 모델과 같은 설정의 새 모델
# print('결정 계수(R^2) = ', r2_score(y_te2, m2.predict(X_te2)))

## [6] 모델 학습
### ① 모델 생성하기 · ② 모델 학습하기

In [ ]:
from sklearn.linear_model import LinearRegression    # 선형 회귀 알고리즘 불러오기
linear_reg = LinearRegression()

**버전 맞춤 · 고치지 않아도 되는 줄** (사본에 더함) — 사이킷런 1.9 이상에서도 교과서와 같은 결과가 나오게 해요. 그냥 실행만 하면 돼요.

In [ ]:
from sklearn.base import clone    # 같은 설정의 모델을 새로 만드는 함수(뒤의 선택 셀에서 사용)
if 'tol' in linear_reg.get_params(): linear_reg.set_params(tol = 0)    # 버전 맞춤

In [ ]:
lin_results = linear_reg.fit(X_train, y_train)

### ③ 가중치와 편향 확인하기

In [ ]:
# 가중치와 편향 확인
w = lin_results.coef_
b = lin_results.intercept_
print(w)
print(b)

In [ ]:
pd.Series(w, index = X.columns)    # (사본에 더함) 가중치를 속성 이름과 나란히 보기

## [7] 성능 평가
### ① 결정 계수 확인하기

In [ ]:
from sklearn.metrics import r2_score
y_pred = lin_results.predict(X_test)
print('결정 계수(R^2) = ', r2_score(y_test, y_pred))

### ② 테스트 데이터로 평가하기

In [ ]:
Y_prediction = y_pred.flatten()
for i in range(6):
    label = y_test.values[i]
    prediction = Y_prediction[i]
    print('실제 기대 수명: {:.3f}, 학습 후 예측 기대 수명: {:.3f}, 차이: {:.3f}'
          .format(label, prediction, label - prediction))

In [ ]:
# (사본에 더함) range(6)을 range(10)으로 바꾸면 교과서 199쪽의 4~6행(8~10번째 행)이 보여요
for i in range(10):
    print(i + 1, '실제: {:.3f}, 예측: {:.3f}, 차이: {:.3f}'.format(y_test.values[i], Y_prediction[i], y_test.values[i] - Y_prediction[i]))

# (사본에 더함) 테스트 데이터 330개 전체의 차이 살펴보기
diff = (y_test.values - y_pred)
print('차이(절댓값)의 평균:', round(np.abs(diff).mean(), 2), '년')
print('차이가 5년 안인 비율:', round((np.abs(diff) <= 5).mean(), 3))

## [8] 수정 및 성능 개선
### ① 선형 회귀 알고리즘 불러오기 · ② 교육 수준(Schooling)으로 학습하기

In [ ]:
lin1 = LinearRegression()
lin1.fit(X_train['Schooling'].values.reshape(-1, 1), y_train.values.reshape(-1, 1))

### ③ 기대 수명 예측값 생성하기

In [ ]:
x_array = np.arange(min(X_train['Schooling']), max(X_train['Schooling'])).reshape(-1, 1)
y_head = lin1.predict(x_array)

### ④ 기대 수명 회귀선 시각화하기

In [ ]:
plt.scatter(X_train['Schooling'], y_train)
plt.plot(x_array, y_head, color = 'red')
plt.xlabel('Schooling')    # 교과서의 'Scooling'은 오타라서 바로잡음
plt.ylabel('Life expectancy')    # 교과서의 'Life excpectancy'도 바로잡음
plt.show()

In [ ]:
# (사본에 더함) 회귀선의 기울기·절편과, 교육 수준 하나만 쓴 모델의 결정 계수
print('기울기:', lin1.coef_, ' 절편:', lin1.intercept_)
print('결정 계수(R^2) = ', r2_score(y_test, lin1.predict(X_test['Schooling'].values.reshape(-1, 1))))

### 더 알아보기 · 속성을 골라 다시 학습하기 (사본에 더한 셀 · 선택)
`cols`에 넣는 속성을 바꿔 가며 결정 계수가 어떻게 달라지는지 비교해 보세요.

In [ ]:
cols = ['Schooling', 'Income composition of resources', 'Adult Mortality', 'HIV/AIDS']
lin2 = clone(linear_reg)    # [6]의 모델과 같은 설정(버전 맞춤 포함)
lin2.fit(X_train[cols], y_train)
print('결정 계수(R^2) = ', r2_score(y_test, lin2.predict(X_test[cols])))

### 더 알아보기 · 결측치가 많은 속성을 먼저 지우고 다시 학습하기 (사본에 더한 셀 · 선택)
앞 단계의 결측치 처리 실험실처럼 `Population`, `Hepatitis B`를 **먼저 지우고** `dropna()`를 하면 남는 데이터가 늘어나요. 지울 속성을 바꿔 가며 결정 계수를 비교해 보세요.

In [ ]:
raw = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u4-structured-1/assets/life-expectancy-data.csv')
df2 = raw.drop(columns = ['Population', 'Hepatitis B']).dropna()    # 지울 속성 바꿔 보기
df2 = df2.replace({'Developing': 0, 'Developed': 1})
df2['Status'] = df2['Status'].astype(int)
X2 = df2.drop(['Life expectancy', 'Country'], axis=1)
y2 = df2['Life expectancy']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size = 0.2, random_state = 96)
m3 = clone(linear_reg).fit(X2_train, y2_train)
print(df2.shape, ' 결정 계수(R^2) = ', r2_score(y2_test, m3.predict(X2_test)))